## Generate Summaries and Tags Using an LLM

## Load useful libraries

In [ ]:
import os
import pickle
import json
from itertools import batched

In [ ]:
from tools_and_shortcuts.ai.llm.OllamaInator import OllamaInator

In [ ]:
from strategic_reports.daily.prompts.prompts import prompt_summarize_and_tag

## User settings

In [ ]:
model_to_use_per_article_summaries_and_tags = 'gpt-oss:120b'
chunk_n = 50
filename_pickled_feeds = os.environ['STRATEGIC_REPORTS_HOME'] + '/output/daily/feed_content.pickled'
filename_json_summaries_and_tags = os.environ['STRATEGIC_REPORTS_HOME'] + '/output/daily/summaries_and_llm_tags.json'

## Initialize the Ollama client

In [ ]:
oi = OllamaInator()

## Load previously retrieved feed content

In [ ]:
with open(filename_pickled_feeds, 'rb') as f:
    feed_list = pickle.load(f)

## QA #1

In [ ]:
qa_title_list = []
for item in feed_list:
    df = item['df']
    for i, row in df.iterrows():
        qa_title_list.append(row['title'].strip())

# This fails when two articles have the same name
#assert len(qa_title_list) == len(list(set(qa_title_list)))

## Iterate through the dataframes and add their contents to a list of dictionaries

In [ ]:
list_content = []
for item in feed_list:
    list_local_df_content = []
    df = item['df']

    if len(df.index > 0):
        list_content.extend(df[['title', 'content', 'link', 'publish_date']].to_dict(orient = 'records'))

# This fails when two articles have the same name
#assert len(qa_title_list) == len(list_content)

## Define a function that (hopefully) fixes some Markdown-related artifacts in response

In [ ]:
def hopefully_fix_the_response(text_that_should_be_json):
    
    if text_that_should_be_json.strip().find('```') == 0:
        text_that_should_be_json = text_that_should_be_json[3:].strip()

    if text_that_should_be_json.strip().find('json') == 0:
        text_that_should_be_json = text_that_should_be_json[4:].strip()
    
    if text_that_should_be_json.split('}')[-1].strip() == '```':
        text_that_should_be_json = '}'.join(text_that_should_be_json.split('}')[0:-1]) + '}'

    return text_that_should_be_json

## Send list of content to summarize and tag to an LLM

In [ ]:
dict_summaries_tags = {'articles' : []}
for i, list_item in enumerate([list(x) for x in batched(list_content, chunk_n)]):
    
    count = -1
    success = False

    while not success:
        count += 1 
        assert count < 5
        
        try:
            response = oi.run_generate(
                prompt_summarize_and_tag + json.dumps({'articles' : list_item}, indent = 4) + '\n',
                model = model_to_use_per_article_summaries_and_tags,
            )
            dict_summaries_tags['articles'].extend(
                json.loads(
                    hopefully_fix_the_response(response['response_text'])
                )['articles']
            )
            success = True
        except:
            success = False
            continue

## Save summarized and tagged content

In [ ]:
with open(filename_json_summaries_and_tags, 'w') as f:
    json.dump(dict_summaries_tags, f)